# 23 · Análisis exploratorio: reoptimizar los hiperparámetros por ganancia sobre `PAQ_P3`

**Etapa 29 del plan.** Los hiperparámetros vigentes salieron del `03`: Optuna por AUC, con 760 columnas, 2 meses de entrenamiento y sin peso en `BAJA+2`. Hoy el modelo (`PAQ_P3`, la vigente del `22`) tiene 1.653 columnas, entrena con 2 a 6 meses y pesa triple a `BAJA+2`. Este notebook busca, con Optuna, los hiperparámetros que hacen ganar más dinero, y los mide en meses que no participaron de la búsqueda.

**Tres fases**, en el banco temporal del `15` (cada modelo entrena desde 201903 hasta dos meses antes del mes evaluado):

| Fase | Meses | Semillas | Para qué |
|---|---|---:|---|
| **Búsqueda** | 201906–08 | 1 | Optuna maximiza la ganancia en la banda, promediada en los 3 meses (60 pruebas) |
| **Reevaluación** | 201906–08 | 4 más | Las 5 mejores pruebas y la vigente se ordenan con 4 semillas que no participaron de la búsqueda |
| **Confirmación** | 201909–10 | 5 | La elegida contra `PAQ_P3`, en meses que no se usaron para elegir. **Decide:** pasa si la diferencia es mayor que 0 y gana en 3 de 5 semillas |

**Qué se busca:** `num_leaves`, `min_data_in_leaf`, `feature_fraction`, `bagging_fraction`, `learning_rate` y el peso de `BAJA+2`, con 296 árboles fijos. Dos pruebas van primero: la vigente y las 400 hojas de 764 filas del `19` (el único empate a favor que quedaba sin combinar).

**La maldición del ganador:** la mejor de muchas pruebas sale inflada en la búsqueda, porque se la eligió por haber tenido suerte en esos meses y con esa semilla. En el `19`, la mejor de 30 sacó +5,34 M en la búsqueda y +0,60 M en la confirmación. Por eso mide la confirmación, no la búsqueda.

**Todo lo de este notebook es exploratorio.** No cambia el resultado de la regla pre-registrada. No carga 201911 ni 201912. **No escribe en `exp/03/` a `exp/22/` ni en los `resultados/03_*` a `22_*`:** solo los lee.

Ficha: `fichas/etapa_29_reoptimizacion.md` (en el laboratorio).

## 1. Parámetros

**Retomar:** si la corrida se corta, se vuelve a correr entera. Optuna sigue desde la última prueba completa, y los modelos ya hechos no se repiten.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import resource
import sys
import time
import warnings

import duckdb
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats
from sklearn.metrics import roc_auc_score

# --- Parámetros -------------------------------------------------------------
BASE_PARQUET = Path('../data/base_2019.parquet')
RESULTADOS = Path('../resultados')
EXP_22 = Path('../exp/22')                          # solo se lee
EXP = Path('../exp/23')
FIGURAS = RESULTADOS / 'figuras'

MES_TOPE = 201910                                   # nada posterior entra a este notebook
MESES_TABLA = [201903, 201904, 201905, 201906, 201907, 201908, 201909, 201910]
MESES_EVALUADOS = [201906, 201907, 201908, 201909, 201910]
MESES_FINALES = [201909, 201910]
MESES_CONSIGNA = [201903, 201904, 201905, 201906]
ULTIMOS = 4
DISTANCIA = 2                                       # la clase de un mes se conoce dos meses después
MESES_TRAIN = MESES_CONSIGNA                        # los pide `construir`; acá cada modelo elige su ventana
MESES_EVAL = MESES_FINALES
ROTOS_EXTRA = [('ctarjeta_master_descuentos', 201910), ('mtarjeta_master_descuentos', 201910)]   # como en el 06 y el 12

CONTROL = 'AC'
BASE_CURADA = 'ACH'
JUNTAS = 'ACH_4'
NUEVAS = {'ACH': {'lags': [], 'tendencia': None, 'curada': True}}
CANDIDATAS = ['ACH_V', 'ACH_Q', 'ACH_T', 'ACH_Z']
REFERENCIA = 'PAQ_P3'                               # la vigente del 22: sus predicciones se leen de exp/22
ELEGIDA = 'ELEGIDA'                                 # la elegida por la regla, en la confirmación
PESO_B2 = 3                                         # el de la vigente
LAGS_MV = [1, 2]                                    # como el 17
ROTO_NUEVO = ('Visa_mfinanciacion_limite', 201904)  # como el 17: solo en las combinaciones
LAGS_R = [1, 2]                                     # como el 20
MESES_BUSQUEDA = [201906, 201907, 201908]
MESES_CONFIRMACION = [201909, 201910]
N_PRUEBAS = 60
N_AL_AZAR = 10                                      # las primeras pruebas de TPE son al azar
N_MEJORES = 5                                       # se reevalúan con las otras semillas
SEMILLAS_MINIMAS = 3                                # regla de acumulación: d > 0 y al menos 3 de 5 semillas a favor
ESTUDIO = '23_paq_p3_ganancia'
ESPACIO = {                                         # (mínimo, máximo, escala logarítmica, entero)
    'num_leaves': (16, 1_024, True, True),
    'min_data_in_leaf': (100, 5_000, True, True),
    'feature_fraction': (0.05, 1.0, False, False),
    'bagging_fraction': (0.3, 1.0, False, False),
    'learning_rate': (0.005, 0.08, True, False),
    'peso_b2': (1.0, 6.0, False, False),
}
ENCOLADA_HOJAS = {'num_leaves': 400, 'min_data_in_leaf': 764}   # el empate a favor del 19
HIPOTESIS = 'Hiperparámetros buscados por ganancia (Optuna sobre 201906-08, reevaluados con 4 semillas) ganan más que los del 03 sobre PAQ_P3'
ENVIOS = [2_000, 3_000, 4_000, 5_000, 6_000, 8_000, 10_000]   # curva por cantidad de envíos, solo para informar
UMBRAL = 1.0                                        # M por mes, fijado en la ficha
FOTOS = 9                                           # 201910 llega hasta 201901

N_CURADAS = 30
MINIMO_PREVIOS = 2
BLOQUE = 20_000                                     # filas por bloque al armar las tablas

SEMILLA_PRIMIGENIA = 979201
N_SEMILLAS = 5
N_BOOTSTRAP = 2_000
BANDA = (3_000, 6_000)                              # ambos incluidos
EPSILON = 1e-12
POSITIVOS = ['BAJA+1', 'BAJA+2']
N_HILOS = 8

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500
LARGO_CURVA = 25_000
# -----------------------------------------------------------------------------

PRED = EXP / 'pred'
VARIABLES_CSV = RESULTADOS / '08_variables.csv'     # solo se lee
for d in (PRED, FIGURAS):
    d.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')  # falso positivo pandas 2.2 + Py 3.14

ESPERADO_FILAS_POR_MES = {
    201901: 123_916, 201902: 125_020, 201903: 125_622, 201904: 126_116, 201905: 126_763,
    201906: 128_370, 201907: 129_853, 201908: 131_775, 201909: 133_419, 201910: 135_633,
}
ESPERADO_TABLA = (1_037_551, 5_042)                 # filas, BAJA+2
ESPERADO_N_VARIABLES = {'AC': 760, 'ACH': 880, 'ACH_4': 1_180, 'MV': 43, 'MVH': 215, 'R': 258}
N_MES = 152
ESPERADO_ROTOS = 23


def primos_6_digitos():
    criba = np.ones(1_000_000, dtype=bool)
    criba[:2] = False
    for i in range(2, 1_001):
        if criba[i]:
            criba[i * i::i] = False
    return np.flatnonzero(criba[100_000:]) + 100_000


TODAS = np.random.default_rng(SEMILLA_PRIMIGENIA).choice(primos_6_digitos(), size=20, replace=False).tolist()[:10]
assert TODAS == [294103, 342929, 161093, 677561, 431077, 330679, 511579, 312773, 118147, 567529]
SEMILLAS = TODAS[:N_SEMILLAS]


def huella_archivos():
    """Control 8: huella de las salidas anteriores, para comprobar que no cambian."""
    archivos = [*RESULTADOS.glob('0[3456789]_*'), *RESULTADOS.glob('1[0123456789]_*'), *RESULTADOS.glob('2[012]_*')]
    return {p.name: hashlib.blake2b(p.read_bytes(), digest_size=16).hexdigest() for p in sorted(archivos)}


def memoria_gb():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return round(r / (1e9 if sys.platform == 'darwin' else 1e6), 1)


HUELLA_INICIAL = huella_archivos()
T_INICIO = time.time()
print('Salidas anteriores registradas:', len(HUELLA_INICIAL))

## 2. Dataset y funciones de evaluación

**Control 1:** filas y `BAJA+2` de 201903 a 201910. **Control 2:** nada posterior a 201910 entra al notebook. La clase de 201911 y 201912 nunca se carga.

In [ ]:
def curva(prob, es_b2):
    orden = np.argsort(-prob, kind='stable')
    return np.cumsum(np.where(es_b2[orden], GANANCIA_ACIERTO, -COSTO_ESTIMULO))[:LARGO_CURVA].astype(np.float64)


def banda(cv):
    return cv[..., BANDA[0] - 1:BANDA[1]].mean(axis=-1)


HIPER = json.loads((RESULTADOS / '03_hiperparametros.json').read_text())

t0 = time.time()
con = duckdb.connect()
base = con.execute(f"""
    select * from read_parquet('{BASE_PARQUET.as_posix()}')
    where foto_mes <= {MES_TOPE}
    order by foto_mes, numero_de_cliente
""").df()
con.close()
assert base['foto_mes'].max() == MES_TOPE                                 # control 2
assert base['foto_mes'].value_counts().sort_index().to_dict() == ESPERADO_FILAS_POR_MES

CLAVES = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
VARIABLES = [c for c in base.columns if c not in CLAVES]
BANDERAS = ['active_quarter', 'cliente_vip', 'internet', 'cdescubierto_preacordado',
            'tcallcenter', 'thomebanking', 'tmobile_app', 'ccajas_transacciones']
ESTADOS = ['tcuentas', 'Visa_status', 'Master_status', 'Visa_delinquency', 'Master_delinquency']
FECHAS = [f'{t}_{c}' for t in ('Visa', 'Master') for c in ('Fvencimiento', 'Finiciomora', 'fultimo_cierre', 'fechaalta')]
EDADES = ['cliente_edad', 'cliente_antiguedad']
MONTOS = [v for v in VARIABLES if v not in BANDERAS + ESTADOS + FECHAS + EDADES]
assert len(VARIABLES) == 152 and len(MONTOS) == 129

rotos = pd.read_csv(RESULTADOS / '01_meses_rotos.csv')[['variable', 'foto_mes']]
assert len(rotos) == ESPERADO_ROTOS
rotos = pd.concat([rotos, pd.DataFrame(ROTOS_EXTRA, columns=['variable', 'foto_mes'])], ignore_index=True)
for v, mes in rotos.itertuples(index=False):
    base.loc[base['foto_mes'] == mes, v] = np.nan

FOTO_MES = base['foto_mes'].to_numpy()
PERIODO = (FOTO_MES // 100 * 12 + FOTO_MES % 100).astype(np.int64)
CLAVE = base['numero_de_cliente'].to_numpy(np.int64) * 100_000 + PERIODO
INDICE = pd.Index(CLAVE)
assert INDICE.is_unique
X_MES = np.asfortranarray(base[VARIABLES].to_numpy(np.float32))
CLASE = base['clase_ternaria'].to_numpy()
del base
gc.collect()
print(f'Dataset cargado en {time.time() - t0:.0f} s')

## 3. Tablas de `ACH_4`

Las mismas columnas de `ACH_4` del `15`: 1.180, el baseline más la historia curada y los cuatro bloques del `14`.

**Control 4:** una columna da lo mismo calculada solo con datos hasta su mes. **Control 6:** fórmulas a mano en 300 clientes.

In [ ]:
SIN_HISTORIA = {'lags': [], 'tendencia': None, 'curada': False}


def receta(config):
    return SIN_HISTORIA if config == CONTROL else NUEVAS[config]


def especificacion(config):
    ex = receta(config)
    e = [('original', v, 0) for v in VARIABLES]
    e += [(op, v, k) for k in (1, 2) for op in ('lag', 'delta') for v in VARIABLES]
    for k in ex['lags']:
        e += [(op, v, k) for op in ('lag', 'delta') for v in MONTOS + BANDERAS]
        e += [(op, v, k) for op in ('lag', 'cambio') for v in ESTADOS]
    if ex['tendencia']:
        h = ex['tendencia']
        e += [(op, v, h) for op in ('pend', 'prom', 'min', 'max') for v in MONTOS]
        e += [('perm', v, h) for v in BANDERAS]
    if ex['curada']:
        assert len(CURADAS) == N_CURADAS
        e += [(op, v, k) for op, k in (('ratio', 3), ('z', 4), ('relmax', 4), ('mes_ratio', 3)) for v in CURADAS]
    return e


def nombre(op, v, k):
    return v if op == 'original' else f'{v}_{op}{k}'


def construir(config, meses=None):
    ex = receta(config)
    h = ex['tendencia']
    h_max = max([2] + ex['lags'] + [h or 0] + ([4] if ex['curada'] else []))
    filas = np.flatnonzero(np.isin(FOTO_MES, meses or MESES_TRAIN + MESES_EVAL))
    mes_fila = FOTO_MES[filas]
    n_train = int(np.isin(mes_fila, MESES_TRAIN).sum())
    assert meses or (np.isin(mes_fila[:n_train], MESES_TRAIN).all() and np.isin(mes_fila[n_train:], MESES_EVAL).all())
    pos = {k: INDICE.get_indexer(CLAVE[filas] - k) for k in range(h_max + 1)}

    espec = especificacion(config)
    ubic = {e: i for i, e in enumerate(espec)}
    F = np.empty((len(filas), len(espec)), dtype=np.float32)
    t = -np.arange((h or 0) + 1, dtype=np.float64)[:, None]
    # Por bloques de filas: escribir columna por columna recorre toda la tabla en cada pasada,
    # y con una tabla de más de ~4,5 GB eso tardó decenas de minutos.
    with np.errstate(invalid='ignore', divide='ignore'):
        for inicio in range(0, len(filas), BLOQUE):
            fb = slice(inicio, inicio + BLOQUE)
            for j, v in enumerate(VARIABLES):
                F[fb, ubic[('original', v, 0)]] = X_MES[filas[fb], j]
                hk = {}
                for k in range(h_max + 1):
                    col = X_MES[pos[k][fb], j]
                    col[pos[k][fb] < 0] = np.nan
                    hk[k] = col
                for k in range(1, h_max + 1):
                    if ('lag', v, k) in ubic:
                        F[fb, ubic[('lag', v, k)]] = hk[k]
                    if ('delta', v, k) in ubic:
                        F[fb, ubic[('delta', v, k)]] = hk[0] - hk[k]
                    if ('cambio', v, k) in ubic:
                        a, b = hk[0], hk[k]
                        F[fb, ubic[('cambio', v, k)]] = np.where(np.isnan(a) | np.isnan(b), np.nan, (a != b).astype(np.float32))
                if h and (('prom', v, h) in ubic or ('perm', v, h) in ubic):
                    S = np.stack([hk[k] for k in range(h + 1)]).astype(np.float64)
                    valido = ~np.isnan(S)
                    cuantos = valido.sum(axis=0)
                    S0 = np.where(valido, S, 0.0)
                    if ('perm', v, h) in ubic:
                        F[fb, ubic[('perm', v, h)]] = np.where(cuantos > 0, S0.sum(axis=0), np.nan)
                    else:
                        prom = S0.sum(axis=0) / cuantos
                        dt = np.where(valido, t - (valido * t).sum(axis=0) / cuantos, 0.0)
                        pend = (dt * (S0 - prom)).sum(axis=0) / (dt ** 2).sum(axis=0)
                        pend[cuantos < 2] = np.nan
                        F[fb, ubic[('pend', v, h)]] = pend
                        F[fb, ubic[('prom', v, h)]] = prom
                        F[fb, ubic[('min', v, h)]] = np.where(cuantos > 0, np.where(valido, S, np.inf).min(axis=0), np.nan)
                        F[fb, ubic[('max', v, h)]] = np.where(cuantos > 0, np.where(valido, S, -np.inf).max(axis=0), np.nan)
                if ('ratio', v, 3) in ubic:
                    a, b, c = (hk[k].astype(np.float64) for k in range(3))
                    media3 = (a + b + c) / 3                  # NA si falta alguna foto
                    ratio = np.where(np.abs(media3) > EPSILON, a / media3, np.nan)
                    F[fb, ubic[('ratio', v, 3)]] = ratio
                    x = a
                    P = np.stack([hk[k] for k in range(1, 5)]).astype(np.float64)
                    valido = ~np.isnan(P)
                    cuantos = valido.sum(axis=0)
                    alcanza = (cuantos >= MINIMO_PREVIOS) & ~np.isnan(x)
                    media = np.where(valido, P, 0.0).sum(axis=0) / cuantos
                    desvio = np.sqrt(np.where(valido, (P - media) ** 2, 0.0).sum(axis=0) / cuantos)
                    maximo = np.where(valido, P, -np.inf).max(axis=0)
                    F[fb, ubic[('z', v, 4)]] = np.where(alcanza & (desvio > EPSILON), (x - media) / desvio, np.nan)
                    F[fb, ubic[('relmax', v, 4)]] = np.where(alcanza & (maximo > EPSILON), x / maximo, np.nan)
    for v in (CURADAS if ex['curada'] else []):           # el percentil necesita el mes entero
        F[:, ubic[('mes_ratio', v, 3)]] = pd.Series(F[:, ubic[('ratio', v, 3)]]).groupby(mes_fila).rank(pct=True).to_numpy(np.float32)
    return {'F': F, 'columnas': [nombre(*e) for e in espec], 'espec': espec, 'filas': filas, 'mes_fila': mes_fila,
            'n_train': n_train, 'lags': sorted({k for op, _, k in espec if op == 'lag'}), 'h': h}


CURADAS = pd.read_csv(VARIABLES_CSV)['variable'].tolist()
assert len(CURADAS) == N_CURADAS and set(CURADAS) <= set(MONTOS)


def cero_fijo(x, mes):
    """Percentil dentro del mes: positivos de 0 a 1, negativos de 0 a -1, y el cero en 0."""
    s = pd.Series(x)
    arriba = s.where(s > 0).groupby(mes).rank(pct=True).fillna(0)
    abajo = (-s).where(s < 0).groupby(mes).rank(pct=True).fillna(0)
    return (arriba - abajo).where(s.notna()).to_numpy()


def bloques_nuevos(filas, X, indice, clave, foto_mes):
    """Cada fila usa solo fotos de su mes o anteriores. El percentil usa solo su mes."""
    pos = [indice.get_indexer(clave[filas] - k) for k in range(FOTOS + 1)]
    mes = foto_mes[filas]
    salida = {b: {} for b in CANDIDATAS}
    with np.errstate(invalid='ignore', divide='ignore'):
        for v in CURADAS:
            j = VARIABLES.index(v)
            H = np.stack([np.where(p >= 0, X[p, j], np.nan) for p in pos]).astype(np.float64)
            a, b, c, d = H[:4]
            dif = np.stack([a - b, b - c, c - d])
            valido = ~np.isnan(dif)
            cuantos = valido.sum(axis=0)
            alcanza = cuantos >= MINIMO_PREVIOS
            media = np.where(valido, dif, 0.0).sum(axis=0) / cuantos
            desvio = np.sqrt(np.where(valido, (dif - media) ** 2, 0.0).sum(axis=0) / cuantos)
            salida['ACH_V'][f'{v}_std3'] = np.stack([a, b, c]).std(axis=0)
            salida['ACH_V'][f'{v}_dstd3'] = np.where(alcanza, desvio, np.nan)
            salida['ACH_Q'][f'{v}_q4'] = (a + b) / 2 - (c + d) / 2
            salida['ACH_Q'][f'{v}_peor3'] = np.where(alcanza, np.where(valido, dif, np.inf).min(axis=0), np.nan)
            salida['ACH_Q'][f'{v}_caidas3'] = np.where(alcanza, (dif < 0).sum(axis=0), np.nan)
            P = H[1:]
            valido = ~np.isnan(P)
            cuantos = valido.sum(axis=0)
            alcanza = (cuantos >= MINIMO_PREVIOS) & ~np.isnan(a)
            maximo = np.where(valido, P, -np.inf).max(axis=0)
            media = np.where(valido, P, 0.0).sum(axis=0) / cuantos
            salida['ACH_T'][f'{v}_relmaxT'] = np.where(alcanza & (maximo > EPSILON), a / maximo, np.nan)
            salida['ACH_T'][f'{v}_relmediaT'] = np.where(alcanza & (np.abs(media) > EPSILON), a / media, np.nan)
            salida['ACH_T'][f'{v}_desdemaxT'] = np.where(alcanza, np.where(np.isnan(H), -np.inf, H).argmax(axis=0), np.nan)
            salida['ACH_Z'][f'{v}_zc1'] = cero_fijo(a - b, mes)
            salida['ACH_Z'][f'{v}_zc2'] = cero_fijo(a - c, mes)
    return {b: (np.column_stack(list(c.values())).astype(np.float32), list(c)) for b, c in salida.items()}


t0 = time.time()
T = construir(BASE_CURADA, meses=MESES_TABLA)
assert T['F'].shape == (ESPERADO_TABLA[0], ESPERADO_N_VARIABLES[BASE_CURADA]), T['F'].shape
assert T['columnas'][:N_MES] == VARIABLES and T['columnas'][ESPERADO_N_VARIABLES[CONTROL]].endswith('_ratio3')
assert (CLASE[T['filas']] == 'BAJA+2').sum() == ESPERADO_TABLA[1]         # control 1
B = bloques_nuevos(T['filas'], X_MES, INDICE, CLAVE, FOTO_MES)
assert {b: v[0].shape[1] for b, v in B.items()} == {'ACH_V': 60, 'ACH_Q': 90, 'ACH_T': 90, 'ACH_Z': 60}

CLIENTE_FILA = CLAVE[T['filas']] // 100_000
CLIENTES, POS_CLIENTE = np.unique(CLIENTE_FILA, return_inverse=True)

# Control 4: sin fuga. Con los datos cortados en el mes, cada columna tiene que dar lo mismo.
for corte in (201905, 201906, 201909):
    hasta = FOTO_MES <= corte
    filas_c = np.flatnonzero(FOTO_MES[hasta] == corte)
    C = bloques_nuevos(filas_c, X_MES[hasta], pd.Index(CLAVE[hasta]), CLAVE[hasta], FOTO_MES[hasta])
    en_mes = T['mes_fila'] == corte
    for b in CANDIDATAS:
        assert np.array_equal(C[b][0], B[b][0][en_mes], equal_nan=True), (b, corte)
    del C

# Control 6: fórmulas a mano
rng = np.random.default_rng(SEMILLA_PRIMIGENIA)
muestra = rng.choice(len(T['filas']), size=300, replace=False)
col = {b: {c: i for i, c in enumerate(B[b][1])} for b in CANDIDATAS}


def igual(bloque, nombre_columna, fila, esperado):
    valor = B[bloque][0][fila, col[bloque][nombre_columna]]
    if esperado is None:
        assert np.isnan(valor), (nombre_columna, fila, valor)
    else:
        np.testing.assert_allclose(valor, esperado, rtol=1e-3, atol=1e-3, err_msg=nombre_columna)


for v in CURADAS[:3]:
    serie = pd.Series(X_MES[:, VARIABLES.index(v)].astype(np.float64), index=INDICE)
    for i in muestra:
        clave = CLAVE[T['filas'][i]]
        y = [serie.get(clave - k, np.nan) for k in range(FOTOS + 1)]
        igual('ACH_V', f'{v}_std3', i, None if np.isnan(y[:3]).any() else float(np.std(y[:3])))
        igual('ACH_Q', f'{v}_q4', i, None if np.isnan(y[:4]).any() else (y[0] + y[1]) / 2 - (y[2] + y[3]) / 2)
        dif = [y[k] - y[k + 1] for k in range(3)]
        hay = [x for x in dif if not np.isnan(x)]
        igual('ACH_V', f'{v}_dstd3', i, float(np.std(hay)) if len(hay) >= 2 else None)
        igual('ACH_Q', f'{v}_peor3', i, min(hay) if len(hay) >= 2 else None)
        igual('ACH_Q', f'{v}_caidas3', i, sum(x < 0 for x in hay) if len(hay) >= 2 else None)
        previos = [x for x in y[1:] if not np.isnan(x)]
        alcanza = len(previos) >= 2 and not np.isnan(y[0])
        igual('ACH_T', f'{v}_relmaxT', i, y[0] / max(previos) if alcanza and max(previos) > EPSILON else None)
        igual('ACH_T', f'{v}_relmediaT', i, y[0] / np.mean(previos) if alcanza and abs(np.mean(previos)) > EPSILON else None)
        igual('ACH_T', f'{v}_desdemaxT', i, int(np.nanargmax(y)) if alcanza else None)
v = CURADAS[0]
en_mes = T['mes_fila'] == 201906
d = T['F'][en_mes, T['columnas'].index(f'{v}_delta1')].astype(np.float64)
z = B['ACH_Z'][0][en_mes, col['ACH_Z'][f'{v}_zc1']]
assert (np.isnan(d) == np.isnan(z)).all() and (np.sign(d[~np.isnan(d)]) == np.sign(z[~np.isnan(z)])).all()
hay = d > 0
np.testing.assert_allclose(z[hay], stats.rankdata(d[hay]) / hay.sum(), rtol=1e-5)
print(f'Tablas y controles 4 y 6 OK en {time.time() - t0:.0f} s · memoria máxima {memoria_gb()} GB')

## 4. Combinaciones de Visa y Master, con su historia

El mismo código del `17` y el `22`: las 43 combinaciones de `951_dataset_epic.r` y su lag y delta de 1 y 2 meses (215 columnas).

**Control 4** y **control 6**, como en el `17`.

In [ ]:
SUMAS = ['mfinanciacion_limite', 'msaldototal', 'msaldopesos', 'msaldodolares', 'mconsumospesos', 'mconsumosdolares',
         'mlimitecompra', 'madelantopesos', 'madelantodolares', 'mpagado', 'mpagospesos', 'mpagosdolares',
         'mconsumototal', 'cconsumos', 'cadelantosefectivo', 'mpagominimo']
MINIMOS = ['Fvencimiento', 'Finiciomora']
MAXIMOS = ['fultimo_cierre', 'fechaalta']
COCIENTES = [('mvr_Master_mlimitecompra', 'Master_mlimitecompra', 'mv_mlimitecompra'),
             ('mvr_Visa_mlimitecompra', 'Visa_mlimitecompra', 'mv_mlimitecompra'),
             ('mvr_msaldototal', 'mv_msaldototal', 'mv_mlimitecompra'),
             ('mvr_msaldopesos', 'mv_msaldopesos', 'mv_mlimitecompra'),
             ('mvr_msaldopesos2', 'mv_msaldopesos', 'mv_msaldototal'),
             ('mvr_msaldodolares', 'mv_msaldodolares', 'mv_mlimitecompra'),
             ('mvr_msaldodolares2', 'mv_msaldodolares', 'mv_msaldototal'),
             ('mvr_mconsumospesos', 'mv_mconsumospesos', 'mv_mlimitecompra'),
             ('mvr_mconsumosdolares', 'mv_mconsumosdolares', 'mv_mlimitecompra'),
             ('mvr_madelantopesos', 'mv_madelantopesos', 'mv_mlimitecompra'),
             ('mvr_madelantodolares', 'mv_madelantodolares', 'mv_mlimitecompra'),
             ('mvr_mpagado', 'mv_mpagado', 'mv_mlimitecompra'),
             ('mvr_mpagospesos', 'mv_mpagospesos', 'mv_mlimitecompra'),
             ('mvr_mpagosdolares', 'mv_mpagosdolares', 'mv_mlimitecompra'),
             ('mvr_mconsumototal', 'mv_mconsumototal', 'mv_mlimitecompra'),
             ('mvr_mpagominimo', 'mv_mpagominimo', 'mv_mlimitecompra')]
ROTOS_MV = {(v, int(m)) for v, m in rotos.itertuples(index=False)} | {ROTO_NUEVO}
assert all(f'{t}_{c}' in VARIABLES for t in ('Master', 'Visa') for c in SUMAS + MINIMOS + MAXIMOS)


def combinaciones(X, foto_mes):
    """Las 43 combinaciones del mes, fila por fila. Solo usan la foto de esa fila."""
    col = lambda v: X[:, VARIABLES.index(v)].astype(np.float64)
    roto = lambda v: np.isin(foto_mes, [m for w, m in ROTOS_MV if w == v])
    s = {}
    ms, vs = col('Master_status'), col('Visa_status')
    m10, v10 = np.where(np.isnan(ms), 10, ms), np.where(np.isnan(vs), 10, vs)
    s['mv_status01'] = np.fmax(ms, vs)
    s['mv_status02'] = ms + vs
    s['mv_status03'] = np.maximum(m10, v10)
    s['mv_status04'] = m10 + v10
    s['mv_status05'] = m10 + 100 * v10
    s['mv_status06'] = np.where(np.isnan(vs), m10, vs)
    s['mv_status07'] = np.where(np.isnan(ms), v10, ms)
    for c in SUMAS:
        s[f'mv_{c}'] = np.nan_to_num(col(f'Master_{c}')) + np.nan_to_num(col(f'Visa_{c}'))
    for c in MINIMOS:
        s[f'mv_{c}'] = np.fmin(col(f'Master_{c}'), col(f'Visa_{c}'))
    for c in MAXIMOS:
        s[f'mv_{c}'] = np.fmax(col(f'Master_{c}'), col(f'Visa_{c}'))
    for c in SUMAS + MINIMOS + MAXIMOS:
        s[f'mv_{c}'][roto(f'Master_{c}') | roto(f'Visa_{c}')] = np.nan
    with np.errstate(invalid='ignore', divide='ignore'):
        for nombre_c, num, den in COCIENTES:
            n = s[num] if num in s else col(num)
            s[nombre_c] = np.where(np.abs(s[den]) > EPSILON, n / s[den], np.nan)
    return np.column_stack(list(s.values())).astype(np.float32), list(s)


def bloque_mv(filas, X, indice, clave, foto_mes):
    """MV: las 43 del mes. MVH: además, lag y delta de 1 y 2 meses. Cada fila usa solo fotos de su mes o anteriores."""
    C, nombres = combinaciones(X, foto_mes)
    H = {}
    for k in [0] + LAGS_MV:
        p = indice.get_indexer(clave[filas] - k)
        H[k] = np.where((p >= 0)[:, None], C[np.maximum(p, 0)], np.float32(np.nan))
    partes, columnas = [H[0]], list(nombres)
    for k in LAGS_MV:
        partes += [H[k], H[0] - H[k]]
        columnas += [f'{n}_lag{k}' for n in nombres] + [f'{n}_delta{k}' for n in nombres]
    MVH = np.hstack(partes).astype(np.float32)
    return {'MV': (MVH[:, :len(nombres)], list(nombres)), 'MVH': (MVH, columnas)}


t0 = time.time()
BMV = bloque_mv(T['filas'], X_MES, INDICE, CLAVE, FOTO_MES)
assert {b: v[0].shape for b, v in BMV.items()} == {b: (ESPERADO_TABLA[0], ESPERADO_N_VARIABLES[b]) for b in BMV}
assert not set(BMV['MVH'][1]) & set(T['columnas'] + [c for b in CANDIDATAS for c in B[b][1]])

# Control 4: sin fuga
for corte in (201905, 201906, 201909):
    hasta = FOTO_MES <= corte
    filas_c = np.flatnonzero(FOTO_MES[hasta] == corte)
    C = bloque_mv(filas_c, X_MES[hasta], pd.Index(CLAVE[hasta]), CLAVE[hasta], FOTO_MES[hasta])['MVH'][0]
    assert np.array_equal(C, BMV['MVH'][0][T['mes_fila'] == corte], equal_nan=True), corte
    del C

# Control 6: fórmulas a mano, con pandas, fila por fila
col_mv = {c: i for i, c in enumerate(BMV['MVH'][1])}
serie = {v: pd.Series(X_MES[:, VARIABLES.index(v)].astype(np.float64), index=INDICE)
         for v in ['Master_msaldototal', 'Visa_msaldototal', 'Master_mlimitecompra', 'Visa_mlimitecompra',
                   'Master_Finiciomora', 'Visa_Finiciomora', 'Master_status', 'Visa_status']}


def a_mano(clave):
    """Valores esperados para una clave cliente-período; NA si la foto no existe."""
    if clave not in INDICE:
        return None
    g = {v: serie[v][clave] for v in serie}
    suma = lambda c: (0 if np.isnan(g[f'Master_{c}']) else g[f'Master_{c}']) + (0 if np.isnan(g[f'Visa_{c}']) else g[f'Visa_{c}'])
    saldo, limite = suma('msaldototal'), suma('mlimitecompra')
    fechas = [x for x in (g['Master_Finiciomora'], g['Visa_Finiciomora']) if not np.isnan(x)]
    m10 = 10 if np.isnan(g['Master_status']) else g['Master_status']
    v10 = 10 if np.isnan(g['Visa_status']) else g['Visa_status']
    return {'mv_msaldototal': saldo, 'mv_mlimitecompra': limite,
            'mvr_msaldototal': saldo / limite if abs(limite) > EPSILON else np.nan,
            'mv_Finiciomora': min(fechas) if fechas else np.nan, 'mv_status05': m10 + 100 * v10}


rng = np.random.default_rng(SEMILLA_PRIMIGENIA + 17)
for i in rng.choice(len(T['filas']), size=300, replace=False):
    clave = CLAVE[T['filas'][i]]
    hoy = a_mano(clave)
    for c, x in hoy.items():
        np.testing.assert_allclose(BMV['MVH'][0][i, col_mv[c]], x, rtol=1e-4, atol=1e-3, err_msg=c)
    for k in LAGS_MV:
        antes = a_mano(clave - k)
        for c in ('mvr_msaldototal', 'mv_msaldototal'):
            lag = np.nan if antes is None else antes[c]
            np.testing.assert_allclose(BMV['MVH'][0][i, col_mv[f'{c}_lag{k}']], lag, rtol=1e-4, atol=1e-3, err_msg=f'{c}_lag{k}')
            np.testing.assert_allclose(BMV['MVH'][0][i, col_mv[f'{c}_delta{k}']], hoy[c] - lag, rtol=1e-4, atol=1e-1, err_msg=f'{c}_delta{k}')

# Meses rotos: la combinación queda vacía
vacia = lambda c, m: np.isnan(BMV['MVH'][0][T['mes_fila'] == m, col_mv[c]]).all()
assert vacia('mv_fultimo_cierre', 201907) and vacia('mv_mfinanciacion_limite', 201904)
assert vacia('mv_mfinanciacion_limite_lag1', 201905) and vacia('mv_mfinanciacion_limite_delta2', 201906)
assert not vacia('mv_mfinanciacion_limite', 201905) and not vacia('mv_fultimo_cierre', 201906)
del serie
gc.collect()
print(f'Combinaciones y controles 4 y 6 OK en {time.time() - t0:.0f} s · memoria máxima {memoria_gb()} GB')

## 5. Cambios relativos de Tomás

El mismo código del `20` y el `22`: `(actual − lag) / (|actual| + |lag|)` para las 129 variables de `MONTOS`, contra 1 y 2 meses atrás (258 columnas).

**Control 4** y **control 6**, como en el `20`.

In [ ]:
# La lista de Tomás, copiada de transforms.numeric_variables
_flags = {'active_quarter', 'cliente_vip', 'internet', 'cdescubierto_preacordado',
          'tcallcenter', 'thomebanking', 'tmobile_app', 'ccajas_transacciones'}
_states = {'tcuentas', 'Visa_status', 'Master_status', 'Visa_delinquency', 'Master_delinquency'}
_dates = {f'{t}_{c}' for t in ('Visa', 'Master') for c in ('Fvencimiento', 'Finiciomora', 'fultimo_cierre', 'fechaalta')}
NUMERICAS_TOMAS = [v for v in VARIABLES if v not in _flags | _states | _dates | {'cliente_edad', 'cliente_antiguedad'}]
assert NUMERICAS_TOMAS == MONTOS and len(MONTOS) == 129


def relativo(a, b):
    """Cambio simétrico acotado en [-1, 1]; dos ceros observados dan cero; un faltante da NaN."""
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    den = np.abs(a) + np.abs(b)
    salida = np.full(a.shape, np.nan)
    valido = np.isfinite(a) & np.isfinite(b)
    salida[valido & (den == 0)] = 0.0
    np.divide(a - b, den, out=salida, where=valido & (den > 0))
    return salida


def bloque_r(F, columnas):
    """Las 258 columnas, desde los valores y los lags de la tabla."""
    ubic = {c: i for i, c in enumerate(columnas)}
    salida = np.empty((F.shape[0], len(MONTOS) * len(LAGS_R)), dtype=np.float32)
    nombres = []
    for k in LAGS_R:
        for v in MONTOS:
            for inicio in range(0, F.shape[0], 200_000):
                fb = slice(inicio, inicio + 200_000)
                salida[fb, len(nombres)] = relativo(F[fb, ubic[v]], F[fb, ubic[f'{v}_lag{k}']])
            nombres.append(f'{v}_rel_sim{k}')
    return salida, nombres


def bloque_r_directo(filas, X, indice, clave):
    """El mismo bloque por otro camino: busca las fotos de 1 y 2 meses antes en el dataset."""
    j = [VARIABLES.index(v) for v in MONTOS]
    actual = X[filas][:, j]
    partes = []
    for k in LAGS_R:
        p = indice.get_indexer(clave[filas] - k)
        previo = np.where((p >= 0)[:, None], X[np.maximum(p, 0)][:, j], np.nan)
        partes.append(relativo(actual, previo).astype(np.float32))
    return np.hstack(partes)


t0 = time.time()
BR = {'R': bloque_r(T['F'], T['columnas'])}
assert BR['R'][0].shape == (ESPERADO_TABLA[0], ESPERADO_N_VARIABLES['R'])
assert not set(BR['R'][1]) & set(T['columnas'] + [c for b in CANDIDATAS for c in B[b][1]])
R_ = BR['R'][0]
assert np.nanmin(R_) >= -1 and np.nanmax(R_) <= 1

# Control 4: sin fuga, y por otro camino
for corte in (201905, 201906, 201909):
    hasta = FOTO_MES <= corte
    filas_c = np.flatnonzero(FOTO_MES[hasta] == corte)
    C = bloque_r_directo(filas_c, X_MES[hasta], pd.Index(CLAVE[hasta]), CLAVE[hasta])
    assert np.array_equal(C, R_[T['mes_fila'] == corte], equal_nan=True), corte
    del C

# Control 6: fórmula a mano, fila por fila, con números de Python
col_r = {c: i for i, c in enumerate(BR['R'][1])}
rng = np.random.default_rng(SEMILLA_PRIMIGENIA + 20)
probar = list(MONTOS[:3]) + rng.choice(MONTOS[3:], size=5, replace=False).tolist()
serie = {v: pd.Series(X_MES[:, VARIABLES.index(v)].astype(np.float64), index=INDICE) for v in probar}
for i in rng.choice(len(T['filas']), size=300, replace=False):
    clave = CLAVE[T['filas'][i]]
    for v in probar:
        a = serie[v][clave]
        for k in LAGS_R:
            b = serie[v].get(clave - k, np.nan)
            if np.isnan(a) or np.isnan(b):
                esperado = None
            elif a == 0 and b == 0:
                esperado = 0.0
            else:
                esperado = (a - b) / (abs(a) + abs(b))
            valor = R_[i, col_r[f'{v}_rel_sim{k}']]
            if esperado is None:
                assert np.isnan(valor), (v, k, i, valor)
            else:
                np.testing.assert_allclose(valor, esperado, rtol=1e-6, atol=1e-7, err_msg=f'{v}_rel_sim{k}')
dos_ceros = (T['F'][:, T['columnas'].index('mpayroll')] == 0) & (T['F'][:, T['columnas'].index('mpayroll_lag1')] == 0)
assert dos_ceros.sum() > 0 and (R_[dos_ceros, col_r['mpayroll_rel_sim1']] == 0).all()
del serie
gc.collect()
faltan = np.isnan(R_).mean(axis=0)
print(f'Cambios relativos y controles 4 y 6 OK en {time.time() - t0:.0f} s · memoria máxima {memoria_gb()} GB')
print(f'Faltantes por columna: mediana {100 * np.median(faltan):.1f} %, máximo {100 * faltan.max():.1f} %')

## 6. Ventanas y controles

**Control 3:** ningún modelo entrena con un mes posterior a dos meses antes del evaluado.

**Control 11:** el paquete tiene 1.653 columnas, sin nombres repetidos, y las primeras 1.180 son las de `ACH_4`.

**Control 5:** `PAQ_P3` de 201906, con la primera semilla, da la predicción del `22`.

In [ ]:
warnings.filterwarnings('ignore', message='Resetting categorical features')
del X_MES                                           # las tablas ya están armadas
gc.collect()

FIJOS = {'objective': 'binary', 'metric': 'auc', 'boosting': 'gbdt', 'max_bin': 31, 'bagging_freq': 1,
         'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_gain_to_split': 0.0, 'feature_pre_filter': False,
         'num_threads': N_HILOS, 'deterministic': True, 'force_col_wise': True, 'verbose': -1}
HIPER = json.loads((RESULTADOS / '03_hiperparametros.json').read_text())
MEJORES = {**FIJOS, **HIPER['params']}
ARBOLES = HIPER['arboles']
assert ARBOLES == 296
BINS = {'max_bin': 31, 'feature_pre_filter': False, 'data_random_seed': SEMILLA_PRIMIGENIA}

ES_B2 = CLASE[T['filas']] == 'BAJA+2'
es_mes = {m: T['mes_fila'] == m for m in MESES_TABLA}
ENTRENA = {m: [x for x in MESES_TABLA if x <= m - DISTANCIA] for m in MESES_EVALUADOS}
assert all(ENTRENA[m] and max(ENTRENA[m]) <= m - DISTANCIA for m in MESES_EVALUADOS)        # control 3
assert set(MESES_BUSQUEDA).isdisjoint(MESES_CONFIRMACION) and set(MESES_BUSQUEDA + MESES_CONFIRMACION) == set(MESES_EVALUADOS)

K = ESPERADO_N_VARIABLES[BASE_CURADA]
BLOQUES = [(T['F'], T['columnas'][:K])] + [B[b] for b in CANDIDATAS] + [BMV['MVH'], BR['R']]
UBICACION = {n: (j, i) for j, (_, nombres) in enumerate(BLOQUES) for i, n in enumerate(nombres)}
COLUMNAS_PAQUETE = [n for _, nombres in BLOQUES for n in nombres]
N_PAQUETE = ESPERADO_N_VARIABLES[JUNTAS] + ESPERADO_N_VARIABLES['MVH'] + ESPERADO_N_VARIABLES['R']
assert len(COLUMNAS_PAQUETE) == len(set(COLUMNAS_PAQUETE)) == len(UBICACION) == N_PAQUETE == 1_653    # control 11
assert COLUMNAS_PAQUETE[:ESPERADO_N_VARIABLES[JUNTAS]] == T['columnas'][:K] + [c for b in CANDIDATAS for c in B[b][1]]


def tabla(columnas, filas):
    """Las columnas pedidas, en ese orden, para las filas de la tabla. Se arma por partes para no duplicar memoria."""
    filas = np.flatnonzero(filas) if filas.dtype == bool else filas
    X = np.empty((len(filas), len(columnas)), dtype=np.float32)
    por_bloque = {}
    for k, n in enumerate(columnas):
        j, i = UBICACION[n]
        por_bloque.setdefault(j, ([], []))
        por_bloque[j][0].append(k)
        por_bloque[j][1].append(i)
    for j, (destino_cols, origen_cols) in por_bloque.items():
        M = BLOQUES[j][0]
        for inicio in range(0, len(filas), 100_000):
            f = filas[inicio:inicio + 100_000]
            X[inicio:inicio + len(f), destino_cols] = M[np.ix_(f, origen_cols)]
    return X


def dataset(meses):
    """El paquete para los meses de entrenamiento, sin pesos: el peso se pone en cada prueba."""
    entrena = np.isin(T['mes_fila'], meses)
    clase = CLASE[T['filas'][entrena]]
    d = lgb.Dataset(tabla(COLUMNAS_PAQUETE, entrena), label=np.isin(clase, POSITIVOS).astype(np.int8),
                    feature_name=list(COLUMNAS_PAQUETE), params=BINS, free_raw_data=True).construct()
    return d, clase


def pesar(d, clase, peso):
    d.set_weight(np.where(clase == 'BAJA+2', peso, 1).astype(np.float32))


def entrenar(d, semilla, cambios):
    return lgb.train({**MEJORES, 'seed': semilla, 'data_random_seed': SEMILLA_PRIMIGENIA, **cambios}, d, num_boost_round=ARBOLES)


def ruta(config, mes, semilla):
    return EXP / 'pred' / config / str(mes) / f'{semilla}.npy'


def guardar(destino, prob):
    assert destino.is_relative_to(EXP), destino
    destino.parent.mkdir(parents=True, exist_ok=True)
    np.save(destino, prob)


print(pd.DataFrame({'meses de entrenamiento': {m: f'{ENTRENA[m][0]}-{ENTRENA[m][-1] % 100:02d}' for m in MESES_EVALUADOS},
                    'filas': {m: int(np.isin(T['mes_fila'], ENTRENA[m]).sum()) for m in MESES_EVALUADOS},
                    'fase': {m: 'búsqueda' if m in MESES_BUSQUEDA else 'confirmación' for m in MESES_EVALUADOS}}).to_string())

# Control 5
marca = EXP / 'control_5.txt'
if marca.exists():
    print('Control 5 ya hecho')
else:
    t0 = time.time()
    d, clase = dataset(ENTRENA[201906])
    pesar(d, clase, PESO_B2)
    prob = entrenar(d, SEMILLAS[0], {}).predict(tabla(COLUMNAS_PAQUETE, es_mes[201906]), num_threads=N_HILOS).astype(np.float32)
    assert np.array_equal(prob, np.load(EXP_22 / 'pred' / REFERENCIA / '201906' / f'{SEMILLAS[0]}.npy')), 'PAQ_P3 no reproduce el 22'
    marca.parent.mkdir(parents=True, exist_ok=True)
    marca.write_text('ok\n')
    del d, clase, prob
    gc.collect()
    print(f'Control 5 OK en {time.time() - t0:.0f} s · memoria máxima {memoria_gb()} GB')
print('Controles 3 y 11 OK')

## 7. Búsqueda

Optuna (TPE, semilla 979201) maximiza la ganancia en la banda de 3.000 a 6.000 envíos, promediada en 201906, 201907 y 201908, con la primera semilla. Las primeras 10 pruebas son al azar. Antes van dos encoladas: **la vigente** y **las 400 hojas de 764 filas** del `19`, con el resto de los hiperparámetros vigentes.

El dataset de cada mes se arma una sola vez; en cada prueba solo cambian los pesos y los hiperparámetros. Cada predicción se guarda en `exp/23/pred/t<número>/`.

**Control 10:** estudio y storage nuevos, `bagging_freq` 1 en todas las pruebas. **Control 5 bis:** la prueba de la vigente da las predicciones del `22` en los 3 meses. **Retomar:** Optuna sigue desde la última prueba completa; si una prueba encolada quedó cortada, se vuelve a encolar.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
EXP.mkdir(parents=True, exist_ok=True)
estudio = optuna.create_study(study_name=ESTUDIO, storage=f'sqlite:///{(EXP / "optuna.db").as_posix()}', direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=SEMILLA_PRIMIGENIA, n_startup_trials=N_AL_AZAR),
                              load_if_exists=True)
assert optuna.study.get_all_study_names(storage=f'sqlite:///{(EXP / "optuna.db").as_posix()}') == [ESTUDIO]   # control 10
assert MEJORES['bagging_freq'] == 1                                                                          # control 10

VIGENTE = {**HIPER['params'], 'peso_b2': float(PESO_B2)}
ENCOLADAS = {'vigente': VIGENTE, '400 hojas del 19': {**VIGENTE, **ENCOLADA_HOJAS}}
for k, (lo, hi, _, _) in ESPACIO.items():
    assert all(lo <= v[k] <= hi for v in ENCOLADAS.values()), k
COMPLETE = optuna.trial.TrialState.COMPLETE
completas = lambda: [t for t in estudio.get_trials(deepcopy=False) if t.state == COMPLETE]
hechas = {t.user_attrs.get('encolada') for t in completas()}
en_cola = {t.user_attrs.get('encolada') for t in estudio.get_trials(deepcopy=False) if t.state == optuna.trial.TrialState.WAITING}
for nombre, params in ENCOLADAS.items():
    if nombre not in hechas and nombre not in en_cola:
        estudio.enqueue_trial(params, user_attrs={'encolada': nombre})
pendientes = N_PRUEBAS - len(completas())
print(f'Pruebas completas: {len(completas())} · pendientes: {max(pendientes, 0)}')


def sugerir(trial):
    return {k: (trial.suggest_int(k, lo, hi, log=lg) if entero else trial.suggest_float(k, lo, hi, log=lg))
            for k, (lo, hi, lg, entero) in ESPACIO.items()}


def separar(params):
    """Los hiperparámetros de LightGBM y el peso de BAJA+2, por separado."""
    p = dict(params)
    return p, p.pop('peso_b2')


if pendientes > 0:
    t0 = time.time()
    D, CL, X = {}, {}, {}
    for m in MESES_BUSQUEDA:
        D[m], CL[m] = dataset(ENTRENA[m])
        X[m] = tabla(COLUMNAS_PAQUETE, es_mes[m])
    print(f'Datasets de la búsqueda en {time.time() - t0:.0f} s · memoria máxima {memoria_gb()} GB', flush=True)

    def objetivo(trial):
        hiper, peso = separar(sugerir(trial))
        t0 = time.time()
        for m in MESES_BUSQUEDA:
            pesar(D[m], CL[m], peso)
            prob = entrenar(D[m], SEMILLAS[0], hiper).predict(X[m], num_threads=N_HILOS).astype(np.float32)
            guardar(ruta(f't{trial.number:03d}', m, SEMILLAS[0]), prob)
            trial.set_user_attr(f'banda_{m}', float(banda(curva(prob, ES_B2[es_mes[m]])) / 1e6))
            trial.set_user_attr(f'auc_{m}', float(roc_auc_score(ES_B2[es_mes[m]], prob)))
        trial.set_user_attr('segundos', round(time.time() - t0, 1))
        valor = float(np.mean([trial.user_attrs[f'banda_{m}'] for m in MESES_BUSQUEDA]))
        print(f'Prueba {trial.number:>2}: {valor:7.2f} M · hojas {hiper["num_leaves"]:>4} de {hiper["min_data_in_leaf"]:>4} filas · '
              f'ff {hiper["feature_fraction"]:.2f} · bf {hiper["bagging_fraction"]:.2f} · lr {hiper["learning_rate"]:.4f} · '
              f'peso {peso:.2f} · {time.time() - t0:.0f} s · memoria máxima {memoria_gb()} GB', flush=True)
        return valor

    estudio.optimize(objetivo, n_trials=pendientes)
    del D, CL, X
    gc.collect()

PRUEBAS = completas()[:N_PRUEBAS]
assert len(PRUEBAS) == N_PRUEBAS                                                         # control 7
NUMERO_VIGENTE = next(t.number for t in PRUEBAS if t.user_attrs.get('encolada') == 'vigente')

for m in MESES_BUSQUEDA:                                                                 # control 5 bis
    assert np.array_equal(np.load(ruta(f't{NUMERO_VIGENTE:03d}', m, SEMILLAS[0])),
                          np.load(EXP_22 / 'pred' / REFERENCIA / str(m) / f'{SEMILLAS[0]}.npy')), m
print('Control 5 bis OK: la prueba de la vigente da las predicciones del 22')

busqueda = pd.DataFrame([{'prueba': t.number, 'encolada': t.user_attrs.get('encolada', ''), **t.params, 'ganancia_M': t.value,
                          **{f'banda_{m}': t.user_attrs[f'banda_{m}'] for m in MESES_BUSQUEDA},
                          **{f'auc_{m}': t.user_attrs[f'auc_{m}'] for m in MESES_BUSQUEDA},
                          'segundos': t.user_attrs['segundos']} for t in PRUEBAS])
busqueda['contra_vigente_M'] = busqueda['ganancia_M'] - busqueda.loc[busqueda['prueba'] == NUMERO_VIGENTE, 'ganancia_M'].item()
busqueda.round(6).to_csv(RESULTADOS / '23_busqueda.csv', index=False)
with pd.option_context('display.width', 250, 'display.max_columns', 30):
    print(busqueda.sort_values('ganancia_M', ascending=False).head(10).round(3).to_string(index=False))
print(f'Búsqueda: {len(PRUEBAS)} pruebas · {busqueda["segundos"].sum() / 60:.0f} min · '
      f'{busqueda["segundos"].mean():.0f} s por prueba (de {busqueda["segundos"].min():.0f} a {busqueda["segundos"].max():.0f})')

## 8. Reevaluación y elección

Las 5 pruebas con más ganancia en la búsqueda (sin contar la vigente) se entrenan con las otras 4 semillas en los mismos 3 meses. La vigente ya tiene esas semillas en el `22`.

**Regla de elección, fijada antes de correr:** gana la de mayor ganancia media con **las 4 semillas que no participaron de la búsqueda**, entre las 5 y la vigente. Si gana la vigente, no hay confirmación y queda `PAQ_P3`. La media con las 5 semillas se informa también.

**Control 13:** la elección se guarda en `exp/23/eleccion.json` antes de la primera predicción de 201909–10. Al retomar, se recalcula y tiene que dar la misma.

In [ ]:
NUEVAS = SEMILLAS[1:]
MEJORES_PRUEBAS = busqueda[busqueda['prueba'] != NUMERO_VIGENTE].nlargest(N_MEJORES, 'ganancia_M')['prueba'].tolist()
PARAMS = {t.number: t.params for t in PRUEBAS}
falta = lambda c, m, s: not ruta(c, m, s).exists()
print('Modelos pendientes de la reevaluación:',
      sum(falta(f't{n:03d}', m, s) for n in MEJORES_PRUEBAS for m in MESES_BUSQUEDA for s in NUEVAS))
segundos = {}
for m in MESES_BUSQUEDA:
    if not any(falta(f't{n:03d}', m, s) for n in MEJORES_PRUEBAS for s in NUEVAS):
        continue
    d, clase = dataset(ENTRENA[m])
    X_m = tabla(COLUMNAS_PAQUETE, es_mes[m])
    for n in MEJORES_PRUEBAS:
        hiper, peso = separar(PARAMS[n])
        pesar(d, clase, peso)
        for s in NUEVAS:
            if falta(f't{n:03d}', m, s):
                t0 = time.time()
                guardar(ruta(f't{n:03d}', m, s), entrenar(d, s, hiper).predict(X_m, num_threads=N_HILOS).astype(np.float32))
                segundos.setdefault('reevaluación', []).append(time.time() - t0)
    print(f'Reevaluación · {m}: {np.mean(segundos.get("reevaluación", [0])):.0f} s por modelo · memoria máxima {memoria_gb()} GB', flush=True)
    del d, clase, X_m
    gc.collect()


def leer(c, m, s):
    if c == REFERENCIA:
        return np.load(EXP_22 / 'pred' / REFERENCIA / str(m) / f'{s}.npy')
    return np.load(ruta(c, m, s))


def bandas_por_semilla(c, meses, semillas):
    return np.array([[banda(curva(leer(c, m, s), ES_B2[es_mes[m]])) / 1e6 for s in semillas] for m in meses])


filas = []
for c, n in [(REFERENCIA, NUMERO_VIGENTE)] + [(f't{n:03d}', n) for n in MEJORES_PRUEBAS]:
    b = bandas_por_semilla(c, MESES_BUSQUEDA, SEMILLAS)
    filas.append({'prueba': n, 'config': c, 'busqueda_M': b[:, 0].mean(), 'reevaluacion_4_M': b[:, 1:].mean(),
                  'cinco_semillas_M': b.mean(), **{f'banda_{m}': b[i, 1:].mean() for i, m in enumerate(MESES_BUSQUEDA)},
                  **PARAMS[n]})
reevaluacion = pd.DataFrame(filas)
for col in ('busqueda_M', 'reevaluacion_4_M', 'cinco_semillas_M'):
    reevaluacion[col.replace('_M', '_contra_vigente_M')] = reevaluacion[col] - reevaluacion.loc[0, col]
reevaluacion.round(6).to_csv(RESULTADOS / '23_reevaluacion.csv', index=False)
with pd.option_context('display.width', 250, 'display.max_columns', 30):
    print(reevaluacion[['prueba', 'busqueda_contra_vigente_M', 'reevaluacion_4_contra_vigente_M', 'cinco_semillas_contra_vigente_M',
                        *ESPACIO]].round(3).to_string(index=False))

ganadora = reevaluacion.loc[reevaluacion['reevaluacion_4_M'].idxmax()]
eleccion = {'prueba': int(ganadora['prueba']), 'es_la_vigente': bool(ganadora['config'] == REFERENCIA),
            'params': PARAMS[int(ganadora['prueba'])], 'reevaluacion_4_contra_vigente_M': float(ganadora['reevaluacion_4_contra_vigente_M']),
            'mejores': MEJORES_PRUEBAS}
ARCHIVO_ELECCION = EXP / 'eleccion.json'
confirmacion_hecha = any(ruta(ELEGIDA, m, s).exists() for m in MESES_CONFIRMACION for s in SEMILLAS)
if ARCHIVO_ELECCION.exists():
    guardada = json.loads(ARCHIVO_ELECCION.read_text())
    assert guardada['prueba'] == eleccion['prueba'] and guardada['mejores'] == eleccion['mejores'], 'La elección cambió al retomar'   # control 13
else:
    assert not confirmacion_hecha                                                                    # control 13
    ARCHIVO_ELECCION.write_text(json.dumps({**eleccion, 'fecha_hora': time.strftime('%Y-%m-%d %H:%M:%S')}))
ELEGIDA_PARAMS = eleccion['params']
print()
print('Elegida:', 'la vigente: no hay confirmación' if eleccion['es_la_vigente'] else
      f'prueba {eleccion["prueba"]}, {eleccion["reevaluacion_4_contra_vigente_M"]:+.2f} M por mes contra la vigente con 4 semillas nuevas')
print({k: round(v, 4) for k, v in ELEGIDA_PARAMS.items()})

## 9. Confirmación

La elegida en 201909 y 201910, con las 5 semillas, contra `PAQ_P3` del `22`. Cada modelo entrena con su ventana: desde 201903 hasta dos meses antes.

**Medida y error:** como en el `22`. Ganancia en la banda; error de clientes (bootstrap pareado) y de semilla (desvío común); el ensamble es el semillerío de las 5.

**Además, sin decidir:** la elegida en los 5 meses (en 201906–08 está inflada por haberla elegido ahí).

In [ ]:
CONTRASTES = [(ELEGIDA, REFERENCIA, MESES_CONFIRMACION), (ELEGIDA, REFERENCIA, MESES_EVALUADOS)]
if eleccion['es_la_vigente']:
    print('Ganó la vigente en la reevaluación: no hay confirmación. Queda PAQ_P3.')
    comparacion = pd.DataFrame()
else:
    hiper, peso = separar(ELEGIDA_PARAMS)
    for m in MESES_CONFIRMACION:
        if not any(falta(ELEGIDA, m, s) for s in SEMILLAS):
            continue
        assert max(ENTRENA[m]) <= m - DISTANCIA                                                    # control 3
        d, clase = dataset(ENTRENA[m])
        pesar(d, clase, peso)
        X_m = tabla(COLUMNAS_PAQUETE, es_mes[m])
        for s in SEMILLAS:
            if falta(ELEGIDA, m, s):
                t0 = time.time()
                guardar(ruta(ELEGIDA, m, s), entrenar(d, s, hiper).predict(X_m, num_threads=N_HILOS).astype(np.float32))
                segundos.setdefault('confirmación', []).append(time.time() - t0)
        print(f'Confirmación · {m}: {np.mean(segundos.get("confirmación", [0])):.0f} s por modelo · memoria máxima {memoria_gb()} GB', flush=True)
        del d, clase, X_m
        gc.collect()
    assert ARCHIVO_ELECCION.stat().st_mtime < min(ruta(ELEGIDA, m, s).stat().st_mtime
                                                  for m in MESES_CONFIRMACION for s in SEMILLAS)   # control 13

    def puntaje(c, m):
        if c == ELEGIDA and m in MESES_BUSQUEDA:
            return np.stack([leer(f't{eleccion["prueba"]:03d}', m, s) for s in SEMILLAS])
        return np.stack([leer(c, m, s) for s in SEMILLAS])

    PUNTAJE = {(c, m): puntaje(c, m) for c in (REFERENCIA, ELEGIDA) for m in MESES_EVALUADOS}
    assert all(v.shape == (N_SEMILLAS, es_mes[k[1]].sum()) and not np.isnan(v).any() for k, v in PUNTAJE.items())   # control 7
    B_MES = {(c, m): np.array([banda(curva(p, ES_B2[es_mes[m]])) / 1e6 for p in PUNTAJE[(c, m)]]) for c, m in PUNTAJE}
    meses_tabla = pd.DataFrame([{'config': c, 'semilla': s, 'foto_mes': m, 'ganancia_banda_M': round(B_MES[(c, m)][i], 4),
                                 'auc': round(roc_auc_score(ES_B2[es_mes[m]], PUNTAJE[(c, m)][i]), 6)}
                                for c, m in PUNTAJE for i, s in enumerate(SEMILLAS)])
    meses_tabla.to_csv(RESULTADOS / '23_meses.csv', index=False)

    posicion = np.arange(1, BANDA[1] + 1)
    PESO = (BANDA[1] - np.maximum(posicion, BANDA[0]) + 1) / (BANDA[1] - BANDA[0] + 1)

    def por_cliente(config, lista_meses):
        """Lo que cada cliente aporta al promedio mensual de la banda."""
        v = np.zeros(len(CLIENTES))
        for m in lista_meses:
            idx = np.flatnonzero(es_mes[m])
            for p in PUNTAJE[(config, m)]:
                orden = np.argsort(-p, kind='stable')[:BANDA[1]]
                np.add.at(v, POS_CLIENTE[idx[orden]], PESO * np.where(ES_B2[idx[orden]], GANANCIA_ACIERTO, -COSTO_ESTIMULO))
        return v / (len(lista_meses) * N_SEMILLAS)

    def remuestrear(d, n_veces, semilla):
        rng = np.random.default_rng(semilla)
        n = len(d)
        con_valor = np.flatnonzero(d)
        salida = np.empty(n_veces)
        for b in range(n_veces):
            veces = np.bincount(rng.integers(0, n, size=n), minlength=n)
            salida[b] = veces[con_valor] @ d[con_valor]
        return salida

    por_semilla = lambda c, lista: np.mean([B_MES[(c, m)] for m in lista], axis=0)
    ensamble = lambda c, lista: float(np.mean([banda(curva(PUNTAJE[(c, m)].mean(axis=0), ES_B2[es_mes[m]])) / 1e6 for m in lista]))

    filas = []
    for j, (c, contra, lista) in enumerate(CONTRASTES):
        d_cliente = por_cliente(c, lista) - por_cliente(contra, lista)
        d = d_cliente.sum() / 1e6
        assert abs(d - (por_semilla(c, lista).mean() - por_semilla(contra, lista).mean())) < 1e-6
        e_c = (remuestrear(d_cliente, N_BOOTSTRAP, SEMILLA_PRIMIGENIA + j) / 1e6).std()
        desvio = np.sqrt((por_semilla(c, lista).var(ddof=1) + por_semilla(contra, lista).var(ddof=1)) / 2)
        e_s = desvio * np.sqrt(2 / N_SEMILLAS)
        e_t = float(np.hypot(e_c, e_s))
        filas.append({'config': c, 'contra': contra, 'meses': f'{lista[0]}-{lista[-1] % 100:02d}', 'decide': lista == MESES_CONFIRMACION,
                      'd_M': d, 'error_clientes_M': e_c, 'error_semilla_M': e_s, 'error_total_M': e_t,
                      'ic95_inf_M': d - 1.96 * e_t, 'ic95_sup_M': d + 1.96 * e_t,
                      'd_ensamble_M': ensamble(c, lista) - ensamble(contra, lista),
                      'semillas_a_favor': int((por_semilla(c, lista) > por_semilla(contra, lista)).sum()),
                      **{f'd_{m}': B_MES[(c, m)].mean() - B_MES[(contra, m)].mean() for m in lista}})
    comparacion = pd.DataFrame(filas)
    comparacion.round(4).to_csv(RESULTADOS / '23_comparacion.csv', index=False)

    print('EXPLORATORIO')
    with pd.option_context('display.width', 250, 'display.max_columns', 30):
        print(comparacion.round(2).to_string(index=False))
    print()
    print('Ganancia en la banda por mes (M, media de semillas):')
    print(pd.DataFrame({c: {m: B_MES[(c, m)].mean() for m in MESES_EVALUADOS} for c in (REFERENCIA, ELEGIDA)}).T.round(2).to_string())
    print()
    print('AUC por mes (media de semillas):')
    print(meses_tabla.groupby(['config', 'foto_mes'])['auc'].mean().unstack().loc[[REFERENCIA, ELEGIDA]].round(4).to_string())
    CURVA = {c: np.mean([curva(p, ES_B2[es_mes[m]]) for m in MESES_CONFIRMACION for p in PUNTAJE[(c, m)]], axis=0) / 1e6
             for c in (REFERENCIA, ELEGIDA)}
    print()
    print('Diferencia por cantidad de envíos contra PAQ_P3 en 201909-10 (M, media de meses y semillas):',
          {n: round(float(CURVA[ELEGIDA][n - 1] - CURVA[REFERENCIA][n - 1]), 2) for n in ENVIOS})

## 10. Cuánto se infló y qué movió la ganancia

- **La inflación:** la diferencia de la elegida contra la vigente en cada fase. Lo esperable es que baje de la búsqueda a la confirmación.
- **El borde:** si la elegida quedó a menos del 5 % de un extremo de algún rango (en la escala de la búsqueda), conviene mirar más allá.
- **Qué movió la ganancia:** cada hiperparámetro contra la ganancia de la búsqueda, en la figura 1.

In [ ]:
fases = {'búsqueda (1 semilla, 201906-08)': float(busqueda.loc[busqueda['prueba'] == eleccion['prueba'], 'contra_vigente_M'].item()),
         'reevaluación (4 semillas nuevas, 201906-08)': eleccion['reevaluacion_4_contra_vigente_M']}
if not comparacion.empty:
    fases['confirmación (5 semillas, 201909-10)'] = float(comparacion.loc[comparacion['decide'], 'd_M'].item())
print('Elegida contra la vigente, por fase (M por mes):', {k: round(v, 2) for k, v in fases.items()})


def lugar(k, v):
    lo, hi, lg, _ = ESPACIO[k]
    return (np.log(v) - np.log(lo)) / (np.log(hi) - np.log(lo)) if lg else (v - lo) / (hi - lo)


bordes = {k: round(float(lugar(k, v)), 3) for k, v in ELEGIDA_PARAMS.items() if lugar(k, v) < 0.05 or lugar(k, v) > 0.95}
print('Hiperparámetros de la elegida en el borde del rango (0 = mínimo, 1 = máximo):', bordes or 'ninguno')
print()
print('Correlación de rangos de cada hiperparámetro con la ganancia de la búsqueda:')
print({k: round(float(stats.spearmanr(busqueda[k], busqueda['ganancia_M']).statistic), 2) for k in ESPACIO})

## 11. Figuras

Tamaño 1920 × 1080 (16:9). Todas llevan el rótulo de exploratorias.

In [ ]:
TINTA_2, GRILLA, AZUL, NARANJA = '#52514e', '#e1e0d9', '#2a78d6', '#d9711c'
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#c3c2b7',
                     'axes.labelcolor': TINTA_2, 'xtick.color': TINTA_2, 'ytick.color': TINTA_2, 'text.color': '#0b0b0b',
                     'axes.grid': True, 'grid.color': GRILLA, 'grid.linewidth': 0.6, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 13, 'axes.titlesize': 14})
TAMANO, DPI = (12.8, 7.2), 150


def coma(x, decimales=1):
    return f'{x:,.{decimales}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


formato = FuncFormatter(lambda v, _: coma(v, 2).rstrip('0').rstrip(','))
ROTULOS = {'num_leaves': 'num_leaves', 'min_data_in_leaf': 'min_data_in_leaf', 'feature_fraction': 'feature_fraction',
           'bagging_fraction': 'bagging_fraction', 'learning_rate': 'learning_rate', 'peso_b2': 'peso de BAJA+2'}

fig, ejes = plt.subplots(2, 3, figsize=TAMANO, sharey=True)
vig = busqueda['prueba'] == NUMERO_VIGENTE
eleg = busqueda['prueba'] == eleccion['prueba']
for ax, k in zip(ejes.flat, ESPACIO):
    ax.scatter(busqueda[k], busqueda['contra_vigente_M'], s=28, color=AZUL, alpha=0.75, label='pruebas')
    ax.scatter(busqueda.loc[vig, k], busqueda.loc[vig, 'contra_vigente_M'], s=110, facecolors='none', edgecolors='#0b0b0b', lw=2, label='vigente')
    if not eleccion['es_la_vigente']:
        ax.scatter(busqueda.loc[eleg, k], busqueda.loc[eleg, 'contra_vigente_M'], s=110, color=NARANJA, label='elegida', zorder=3)
    ax.axhline(0, color=TINTA_2, lw=0.8)
    if ESPACIO[k][2]:
        ax.set_xscale('log')
    else:
        ax.xaxis.set_major_formatter(formato)
    ax.set_xlim(ESPACIO[k][0], ESPACIO[k][1])
    ax.set_xlabel(ROTULOS[k])
    ax.yaxis.set_major_formatter(formato)
for ax in ejes[:, 0]:
    ax.set_ylabel('Contra la vigente (M$ por mes)')
ejes.flat[0].legend(loc='lower left', fontsize=11, frameon=False)
fig.suptitle(f'Exploratorio: la búsqueda de Optuna, {len(PRUEBAS)} pruebas (201906-08, una semilla)', x=0.02, ha='left', fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURAS / '23_1_busqueda.png', dpi=DPI)
plt.show()

fig, ax = plt.subplots(figsize=TAMANO)
x = list(fases)
ax.axhline(0, color=TINTA_2, lw=1.2)
ax.bar(x, list(fases.values()), color=AZUL, width=0.5)
if not comparacion.empty:
    r = comparacion.loc[comparacion['decide']].iloc[0]
    ax.errorbar([x[-1]], [r.d_M], yerr=[[r.d_M - r.ic95_inf_M], [r.ic95_sup_M - r.d_M]], fmt='o', color='#0b0b0b', ms=8, capsize=6, lw=2)
ax.yaxis.set_major_formatter(formato)
ax.set_ylabel('Elegida − vigente (M$ por mes)')
ax.grid(axis='x', visible=False)
ax.set_axisbelow(True)
fig.suptitle('Exploratorio: la elegida contra la vigente, en cada fase', x=0.02, ha='left', fontsize=16)
ax.set_title('La confirmación lleva el IC 95 %. Decide: diferencia mayor que 0 y 3 de 5 semillas a favor', loc='left', color=TINTA_2, fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURAS / '23_2_confirmacion.png', dpi=DPI)
plt.show()

## 12. Lectura fijada antes de correr

Decide la elegida contra `PAQ_P3`, en 201909–10 con 5 semillas, con `d` = la diferencia media en la banda. Usa la **regla de acumulación** (decisión del usuario del 2026-11-09, que reemplaza al umbral de ±1 M): como 201909 y 201910 no participan de la búsqueda ni de la elección, alcanza con que la diferencia sea positiva.

| Resultado | Lectura |
|---|---|
| `d` mayor que 0 **y** al menos 3 de 5 semillas a favor | Suma: la elegida pasa a ser la vigente |
| `d` de −1 M o menos | Resta: queda `PAQ_P3` |
| Cualquier otro caso | Empate: queda `PAQ_P3` |

Si en la reevaluación gana la vigente, no hay confirmación y queda `PAQ_P3`. El intervalo se informa siempre.

**Control 9:** las salidas anteriores no cambiaron.

In [ ]:
assert huella_archivos() == HUELLA_INICIAL, 'Cambiaron las salidas anteriores'   # control 9
print(f'Control de salidas OK · {(time.time() - T_INICIO) / 60:.1f} min en total · memoria máxima {memoria_gb()} GB')
if segundos:
    print('Segundos por modelo:', {c: round(float(np.mean(v)), 1) for c, v in segundos.items()})
print()
print('EXPLORATORIO: no cambia el resultado de la regla pre-registrada.')
print()

lectura = lambda d, a_favor: 'Suma' if d > 0 and a_favor >= SEMILLAS_MINIMAS else ('Resta' if d <= -UMBRAL else 'Empate')
if comparacion.empty:
    resultado, d_decide = 'Gana la vigente en la reevaluación', np.nan
    print('Lectura: queda PAQ_P3. Ninguna de las 5 mejores le ganó a la vigente con 4 semillas nuevas.')
else:
    for r in comparacion.itertuples():
        marca = ' · DECIDE' if r.decide else ' · inflada en 201906-08'
        print(f'Elegida (prueba {eleccion["prueba"]}) − PAQ_P3, {r.meses}: {r.d_M:+.2f} M ({r.ic95_inf_M:.2f} a {r.ic95_sup_M:.2f}) · '
              f'error {r.error_total_M:.2f} M · ensamble {r.d_ensamble_M:+.2f} M · semillas a favor {r.semillas_a_favor} de {N_SEMILLAS} · '
              f'{lectura(r.d_M, r.semillas_a_favor) if r.decide else "no decide"}{marca}')
    fila = comparacion.loc[comparacion['decide']].iloc[0]
    d_decide = float(fila['d_M'])
    resultado = lectura(d_decide, int(fila['semillas_a_favor']))

pd.DataFrame([{'config': f'prueba {eleccion["prueba"]}', 'hipotesis': HIPOTESIS, 'contra': REFERENCIA,
               'hiperparametros': json.dumps({k: round(v, 4) for k, v in ELEGIDA_PARAMS.items()}),
               'd_M': round(d_decide, 3), 'resultado': resultado, 'decide': True}]).to_csv(RESULTADOS / '23_iteraciones.tsv', sep='\t', index=False)